# 02 - Split, six-model comparison, ablations
Needs the output of notebook 01 (`urls_features.parquet`, or `urls_merged.csv` as a fallback). Runtime: roughly 10-20 minutes.

**Design, from the data audit**
- **Train / val / test = `kaggle_main` only**, split **by registered domain** (no domain on two sides), domains capped at 50 URLs per class.
- `kaggle_balanced` (84.6% of it duplicated main) and `urlhaus` are **never trained on** (unless you choose the URLhaus time split below). They are external sets, reported as **recall** (malicious sets) and **specificity** (benign set). Their class mix is arbitrary, so precision/accuracy on them would mislead.
- All six models see the **same** capped training sample and the **same** evaluation samples.

In [ ]:
import os, sys, glob, zipfile

base = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())

# Unpack zip if found
for r in [base, "/content", ".", "/kaggle/input"]:
    for pat in ["*url-guard*.zip", "url-guard.zip"]:
        hits = glob.glob(os.path.join(r, pat))
        if hits:
            z = sorted(hits, key=os.path.getmtime)[-1]
            print("Unpacking:", z)
            zipfile.ZipFile(z).extractall(base)
            break

# Ensure src is on sys.path
for d in [os.path.join(base, "url-guard", "src"), os.path.join(base, "src"), "src", "/content/url-guard/src"]:
    if os.path.isdir(d) and d not in sys.path:
        sys.path.insert(0, d)
        print("Added to path:", d)

if os.path.isdir(os.path.join(base, "url-guard")):
    os.chdir(os.path.join(base, "url-guard"))

print("project dir:", os.getcwd())


In [ ]:
%pip install -q tldextract pyarrow

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import experiment as ex, plots
from features import extract_features_df, feature_names

def locate(name, *alts):
    candidates = [name] + list(alts)
    for c in candidates:
        for prefix in [".", "data/processed", "/content", "/content/url-guard/data/processed", "/kaggle/working"]:
            p = os.path.join(prefix, c)
            if os.path.isfile(p):
                return p
        for prefix in ["/content", ".", "/kaggle/working"]:
            hits = glob.glob(os.path.join(prefix, "**", c), recursive=True)
            if hits:
                return hits[0]
    return None

P = locate("urls_features.parquet")
if P:
    print("Loading feature parquet from:", P)
    df = pd.read_parquet(P)
else:
    M_ = locate("urls_merged.csv")
    assert M_, "Could not find urls_features.parquet or urls_merged.csv! Please run notebook 01 first."
    print("Parquet not found, computing features from:", M_)
    df = pd.read_csv(M_)
    df = pd.concat([df, extract_features_df(df["url"])], axis=1).dropna(subset=feature_names())

if "dateadded" in df:
    df["dateadded"] = pd.to_datetime(df["dateadded"], errors="coerce")
print("Dataset shape:", df.shape)
df.groupby(["source", "label"]).size().unstack(fill_value=0)


## 1. Build the splits
Set `URLHAUS_TRAIN_FRAC = 0.7` and re-run from here if the main-only model catches little of URLhaus (see section 3). That moves the **oldest** 70% of URLhaus into training and keeps the **newest** 30% as the fresh test.

In [ ]:
CAP = 50
URLHAUS_TRAIN_FRAC = 0.0
mm = df[df.source == "kaggle_main"]
print("most frequent domains per class in main (before capping):")
for lab in (0, 1): print(" label", lab, mm[mm.label == lab].domain.value_counts().head(5).to_dict())
splits = ex.make_splits(df, cap=CAP, urlhaus_train_frac=URLHAUS_TRAIN_FRAC)
splits.summary()

## 2. Six models, one split

In [ ]:
comp = ex.run_comparison(splits, train_cap=120_000, eval_cap=30_000)
tbl = ex.summary_table(comp)
tbl

In [ ]:
os.makedirs("outputs", exist_ok=True)
plots.plot_confusion_grid(comp.results, "test", "outputs/confusion_test.png"); plt.show()
plots.plot_metric_panels(tbl, "outputs/model_comparison.png"); plt.show()

## 3. Where does each model fail?
Values are the **share flagged malicious**. For malicious groups that is recall (higher is better). For **benign** groups it is the false-positive rate (lower is better).
Rows to read: `subtype` (phishing / malware / defacement recall), `has_ip_host` (URLhaus IP-address malware vs domain-hosted), `domain_seen_in_train` vs unseen domain (the honest "unseen URL" number).

In [ ]:
b = comp.breakdown
piv = b.pivot_table(index=["eval_set", "by", "group"], columns="model", values="flagged_malicious_rate")[ex.MODEL_ORDER]
piv.insert(0, "n", b.groupby(["eval_set", "by", "group"]).n.first())
piv.round(3)

## 4. Does feature engineering help? (question 5)
Same models, same splits; only the feature set changes. Logistic Regression (simple) and Random Forest (flexible), so the answer is not tied to one model.

In [ ]:
ab = ex.feature_set_ablation(splits, ex.FEATURE_SETS, train_cap=100_000, eval_cap=20_000)
ab.round(4)

## 5. Does URL length matter? (question 4)
Other features correlate with length (character counts, ratios), so dropping only the length group can understate its effect. Read all three rows together.

In [ ]:
print(df[df.source == "kaggle_main"].groupby("label").url_length.describe(percentiles=[.5, .9]).round(1))
ab_len = ex.feature_set_ablation(splits, ex.LENGTH_SETS, train_cap=100_000, eval_cap=20_000)
ab_len.round(4)

## 6. Most predictive features (question 2)

In [ ]:
rf = comp.models["Random Forest"]
imp = ex.feature_importance(rf, splits.test, comp.cols)
plots.plot_importance(imp, 15, "outputs/feature_importance.png"); plt.show()
imp.head(15)[["feature", "perm_importance", "impurity_importance", "meaning"]].round(4)

In [ ]:
import shutil
tbl.to_csv("outputs/summary_table.csv", index=False)
comp.results.to_csv("outputs/results_long.csv", index=False)
comp.breakdown.to_csv("outputs/breakdown.csv", index=False)
splits.summary().to_csv("outputs/split_summary.csv", index=False)
ab.to_csv("outputs/ablation_feature_sets.csv", index=False)
ab_len.to_csv("outputs/ablation_length.csv", index=False)
imp.to_csv("outputs/feature_importance.csv", index=False)
print(shutil.make_archive("outputs", "zip", "outputs"))

## Paste back
1. `splits.summary()` table
2. the six-model `tbl`
3. the section-3 breakdown table
4. the two ablation tables
5. top-15 of `imp`

Also download `outputs.zip` (charts + CSVs).